# BERT
End-to-end fine-tuning of RobBERT for multi-label rechtsfeitcodes classification.

**All hyperparameters live in `bert_config.json` or sweeps folder — edit these files, not this notebook.**

Pipeline:
1. Load shared config, frozen split, frozen label space
2. Pre-tokenize documents once
3. Fine-tune RobBERT + classifier head end-to-end
4. Per-label threshold optimisation on val set
5. Evaluate on test set via shared harness
6. Write `artifacts/predictions/{run_name}.parquet`

**Sweep mode:** set `SWEEP_MODE = True` to loop over all configs in `sweeps/`.
Set `SWEEP_MODE = False` to run only `bert_config.json`.

## 0. Imports & shared setup

In [ ]:
import os
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN is not set. Add it to your .env file or export it in the terminal."
    )
login(token=HF_TOKEN)

In [ ]:
import sys
sys.path.insert(0, "..")

import time
import traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

from shared.config  import DEVICE, ARTIFACTS, RESULTS, ROOT, TRAIN_PATH, TEST_PATH
from shared.config  import load_bert_config, BERT_CFG
from shared.data    import load_train_test
from shared.labels  import get_mlb
from shared.splits  import load_split
from shared.harness import evaluate
from shared.io      import write_predictions

print(f"Device: {DEVICE}")

In [5]:
# --- sweep settings ---
# SWEEP_MODE = True    →  loop over configs in sweeps/ (filtered by RUN_ONLY)
# SWEEP_MODE = False   →  single run using sweeps/robbert.json (the default)
# RUN_ONLY examples:
#   None                   → all configs
#   ["eurobert.json"]      → EuroBERT only
#   ["robbert.json", "focal.json"]  → two specific configs
SWEEP_MODE = True
SWEEPS_DIR = ROOT / "sweeps"
RUN_ONLY   = None   # None = all configs; list of filenames to run a subset

## 1. Load data, frozen split, frozen label space

In [ ]:
train_full_df, test_df = load_train_test(TRAIN_PATH, TEST_PATH)
mlb, classes, num_classes = get_mlb()
train_idx, val_idx = load_split()

train_df = train_full_df.iloc[train_idx].reset_index(drop=True)
val_df   = train_full_df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

## 2. Binarize labels

In [ ]:
y_train = mlb.transform(train_df["rechtsfeitcodes"]).astype(np.float32)
y_val   = mlb.transform(val_df["rechtsfeitcodes"]).astype(np.float32)
y_test  = mlb.transform(test_df["rechtsfeitcodes"]).astype(np.float32)

print(f"y_train : {y_train.shape}")
print(f"y_val   : {y_val.shape}")
print(f"y_test  : {y_test.shape}")
print(f"Labels with 0 train support : {(y_train.sum(axis=0) == 0).sum()} / {num_classes}")

## 3. Helper classes and functions

In [ ]:
def pretokenize(texts: list, cfg, tokenizer) -> list:
    """
    Tokenize and chunk all documents once upfront.
    Returns a list of (input_ids_tensor, attention_mask_tensor) per document.
    Called once per split — not repeated every epoch.
    """
    CLS_ID = tokenizer.cls_token_id
    SEP_ID = tokenizer.sep_token_id
    PAD_ID = tokenizer.pad_token_id

    all_chunks = []
    for text in tqdm(texts, desc="Pre-tokenizing"):
        input_ids = tokenizer(
            text, return_tensors="pt", truncation=False
        )["input_ids"][0].tolist()

        tokens = input_ids[1:-1]
        stride = cfg.stride if cfg.stride > 0 else cfg.content_size
        starts = list(range(0, max(len(tokens), 1), stride))
        chunks = [tokens[s : s + cfg.content_size] for s in starts]
        chunks = chunks[:cfg.max_chunks]

        all_ids, all_masks = [], []
        for chunk in chunks:
            ids     = [CLS_ID] + chunk + [SEP_ID]
            mask    = [1] * len(ids)
            pad_len = cfg.content_size + 2 - len(ids)
            ids    += [PAD_ID] * pad_len
            mask   += [0]      * pad_len
            all_ids.append(ids)
            all_masks.append(mask)

        all_chunks.append((
            torch.tensor(all_ids,   dtype=torch.long),
            torch.tensor(all_masks, dtype=torch.long),
        ))
    return all_chunks


class MultiLabelDocumentDataset(Dataset):
    """Serves pre-tokenized chunks — no re-tokenization per epoch."""
    def __init__(self, chunks: list, labels: np.ndarray):
        self.chunks = chunks
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        input_ids, attention_mask = self.chunks[idx]
        return {
            "input_ids"      : input_ids,
            "attention_mask" : attention_mask,
            "labels"         : self.labels[idx],
        }


def collate_fn(batch):
    """Pad all documents in the batch to the same number of chunks."""
    seq_len   = batch[0]["input_ids"].shape[1]
    max_chunks = max(item["input_ids"].shape[0] for item in batch)
    PAD_ID     = 1
    input_ids_batch, attention_mask_batch, labels_batch, n_chunks_batch = [], [], [], []

    for item in batch:
        n_chunks = item["input_ids"].shape[0]
        pad      = max_chunks - n_chunks
        ids_pad  = torch.full((pad, seq_len), PAD_ID, dtype=torch.long)
        mask_pad = torch.zeros((pad, seq_len), dtype=torch.long)

        input_ids_batch.append(torch.cat([item["input_ids"],       ids_pad],  dim=0))
        attention_mask_batch.append(torch.cat([item["attention_mask"], mask_pad], dim=0))
        labels_batch.append(item["labels"])
        n_chunks_batch.append(n_chunks)

    return {
        "input_ids"      : torch.stack(input_ids_batch),
        "attention_mask" : torch.stack(attention_mask_batch),
        "labels"         : torch.stack(labels_batch),
        "n_chunks"       : torch.tensor(n_chunks_batch),
    }


class FineTunedMultiLabelClassifier(nn.Module):
    """
    Full RobBERT + linear classifier head.
    Pooling strategy controlled by cfg.pooling: 'cls' | 'mean' | 'max'.
    Loss controlled by cfg.loss_type: 'bce' | 'focal'.
    """
    def __init__(self, num_classes: int, y_train: np.ndarray, cfg):
        super().__init__()
        self.cfg        = cfg
        self.robbert    = AutoModel.from_pretrained(cfg.model_name, attn_implementation="sdpa")
        self.classifier = nn.Linear(self.robbert.config.hidden_size, num_classes)
        self.dropout    = nn.Dropout(0.1)

        if cfg.gradient_checkpointing:
            self.robbert.gradient_checkpointing_enable()

        if cfg.loss_type == "bce":
            pos_counts = y_train.sum(axis=0)
            neg_counts = len(y_train) - pos_counts
            pos_weight = torch.tensor(
                np.clip(neg_counts / np.clip(pos_counts, 1, None), 1, cfg.pos_weight_cap),
                dtype=torch.float32
            )
            self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        elif cfg.loss_type == "focal":
            self.gamma   = cfg.focal_gamma
            self.loss_fn = None
        else:
            raise ValueError(f"Unknown loss_type: {cfg.loss_type}")

    def _pool(self, last_hidden_state, attention_mask, n_real_chunks):
        B = last_hidden_state.shape[0]
        doc_embeddings = []
        for i in range(B):
            real = last_hidden_state[i, :n_real_chunks[i]]
            mask = attention_mask[i, :n_real_chunks[i]]
            if self.cfg.pooling == "cls":
                doc_embeddings.append(real[:, 0, :].max(dim=0).values)
            elif self.cfg.pooling == "mean":
                mask_exp = mask.unsqueeze(-1).float()
                summed   = (real * mask_exp).sum(dim=(0, 1))
                count    = mask_exp.sum(dim=(0, 1)).clamp(min=1)
                doc_embeddings.append(summed / count)
            elif self.cfg.pooling == "max":
                mask_exp = mask.unsqueeze(-1).bool()
                real_masked = real.masked_fill(~mask_exp, float("-inf"))
                doc_embeddings.append(real_masked.flatten(0, 1).max(dim=0).values)
        return torch.stack(doc_embeddings)

    def _focal_loss(self, logits, labels):
        bce   = nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        probs = torch.sigmoid(logits)
        pt    = torch.where(labels == 1, probs, 1 - probs)
        return ((1 - pt) ** self.gamma * bce).mean()

    def forward(self, input_ids, attention_mask, n_chunks, labels=None):
        B, max_chunks, seq_len = input_ids.shape
        ids_flat  = input_ids.view(B * max_chunks, seq_len)
        mask_flat = attention_mask.view(B * max_chunks, seq_len)
        outputs   = self.robbert(input_ids=ids_flat, attention_mask=mask_flat)
        hs_flat   = self.dropout(outputs.last_hidden_state)
        hs_all    = hs_flat.view(B, max_chunks, seq_len, -1)
        doc_emb   = self._pool(hs_all, attention_mask, n_chunks)
        logits    = self.classifier(doc_emb)
        loss = None
        if labels is not None:
            if self.cfg.loss_type == "bce":
                self.loss_fn.pos_weight = self.loss_fn.pos_weight.to(logits.device)
                loss = self.loss_fn(logits, labels)
            else:
                loss = self._focal_loss(logits, labels)
        return {"loss": loss, "logits": logits}


def compute_metrics(eval_pred):
    """Flat 0.5 threshold — used during training for epoch-level monitoring only."""
    logits, labels = eval_pred
    preds  = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)
    active = labels.sum(axis=0) > 0
    return {
        "micro_f1" : f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1" : f1_score(labels[:, active], preds[:, active], average="macro", zero_division=0),
    }

print("Helper classes and functions defined.")

## 4. Training function

In [ ]:
def run_one(cfg, run_name: str = "bert") -> dict:
    """
    Full training + evaluation pipeline for one config.
    Writes metrics to results/, predictions to artifacts/predictions/.
    Returns the metrics dict.
    """
    print(f"\n{'='*60}")
    print(f"RUN: {run_name} — {cfg.model_name}")
    print(f"pooling={cfg.pooling} | loss={cfg.loss_type} | chunks={cfg.max_chunks} | batch={cfg.effective_batch}")
    print(f"{'='*60}")

    # sampling
    if cfg.is_test_run:
        n = cfg.sample_size
        _train_df = train_df.iloc[:n].reset_index(drop=True)
        _val_df   = val_df.iloc[:n//5].reset_index(drop=True)
        _y_train  = y_train[:n]
        _y_val    = y_val[:n//5]
        print(f"TEST RUN — {n} train / {n//5} val docs")
    else:
        _train_df, _val_df = train_df, val_df
        _y_train,  _y_val  = y_train,  y_val
        print("FULL RUN")

    # tokenizer + pre-tokenize once
    tokenizer    = AutoTokenizer.from_pretrained(cfg.model_name)
    train_chunks = pretokenize(_train_df["text"].tolist(), cfg, tokenizer)
    val_chunks   = pretokenize(_val_df["text"].tolist(),   cfg, tokenizer)
    test_chunks  = pretokenize(test_df["text"].tolist(),   cfg, tokenizer)

    train_dataset = MultiLabelDocumentDataset(train_chunks, _y_train)
    val_dataset   = MultiLabelDocumentDataset(val_chunks,   _y_val)
    test_dataset  = MultiLabelDocumentDataset(test_chunks,  y_test)

    # model
    clf = FineTunedMultiLabelClassifier(num_classes=num_classes, y_train=_y_train, cfg=cfg)
    print(f"Trainable params: {sum(p.numel() for p in clf.parameters() if p.requires_grad):,}")

    # training
    training_args = TrainingArguments(
        output_dir                  = str(ARTIFACTS / "models" / run_name),
        num_train_epochs            = cfg.epochs,
        per_device_train_batch_size = cfg.batch_size,
        per_device_eval_batch_size  = cfg.batch_size,
        gradient_accumulation_steps = cfg.grad_accum,
        learning_rate               = cfg.lr,
        warmup_ratio                = cfg.warmup_ratio,
        weight_decay                = cfg.weight_decay,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "micro_f1",
        greater_is_better           = True,
        fp16                        = torch.cuda.is_available(),
        logging_steps               = 50,
        report_to                   = "none",
        dataloader_num_workers      = 0,
        remove_unused_columns       = False,
    )

    trainer = Trainer(
        model           = clf,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        data_collator   = collate_fn,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
    )

    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0

    # threshold optimisation on val
    val_out    = trainer.predict(val_dataset)
    val_probs  = 1 / (1 + np.exp(-val_out.predictions))
    val_labels = val_out.label_ids
    thresholds = np.full(num_classes, 0.5)
    for i in range(num_classes):
        if val_labels[:, i].sum() < cfg.thresh_min_support:
            continue
        best_f1, best_t = 0.0, 0.5
        for t in np.arange(0.1, 0.75, 0.05):
            f1 = f1_score(val_labels[:, i], (val_probs[:, i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[i] = best_t

    # test evaluation
    test_out    = trainer.predict(test_dataset)
    test_probs  = 1 / (1 + np.exp(-test_out.predictions))
    test_labels = test_out.label_ids
    preds_tuned = (test_probs > thresholds[np.newaxis, :]).astype(int)
    preds_flat  = (test_probs > 0.5).astype(int)

    print("--- Tuned thresholds ---")
    metrics_tuned = evaluate(test_labels, preds_tuned, classes, method=run_name)
    print("\n--- Flat 0.5 (reference) ---")
    metrics_flat  = evaluate(test_labels, preds_flat,  classes, method=f"{run_name}_flat", save=False)

    # write val predictions (needed by orchestration to fit routing on val)
    val_predicted = (val_probs > thresholds[np.newaxis, :]).astype(bool)
    write_predictions(
        akteIds   = _val_df["akteId"].tolist(),
        scores    = val_probs,
        predicted = val_predicted,
        classes   = classes,
        method    = f"{run_name}_val",
    )

    # save per-class thresholds so calibration / inference can reload them
    thresh_dir = ARTIFACTS / "models" / run_name
    thresh_dir.mkdir(parents=True, exist_ok=True)
    import json as _json
    with open(thresh_dir / "thresholds.json", "w", encoding="utf-8") as _fh:
        _json.dump({str(c): float(thresholds[i]) for i, c in enumerate(classes)}, _fh)

    # write test predictions
    write_predictions(
        akteIds   = test_df["akteId"].tolist(),
        scores    = test_probs,
        predicted = preds_tuned.astype(bool),
        classes   = classes,
        method    = run_name,
    )

    # write SUMMARY.md section
    curve_rows = []
    for log in trainer.state.log_history:
        if "eval_micro_f1" in log:
            curve_rows.append(
                f"| {int(log['epoch'])} "
                f"| {log.get('loss', float('nan')):.3f} "
                f"| {log['eval_loss']:.3f} "
                f"| {log['eval_micro_f1']:.4f} "
                f"| {log['eval_macro_f1']:.4f} |"
            )

    pc       = pd.DataFrame(metrics_tuned["per_class"])
    active   = pc[pc["support"] > 0].sort_values("f1")
    bottom10 = active.head(10)[["code","f1","support"]].to_markdown(index=False)
    top10    = active.tail(10)[["code","f1","support"]].to_markdown(index=False)

    summary = f"""
## {run_name} — {cfg.model_name}

**Config:** pooling={cfg.pooling} | loss={cfg.loss_type} | max_chunks={cfg.max_chunks} | epochs={cfg.epochs} | lr={cfg.lr} | batch={cfg.effective_batch}

**Training time:** {elapsed/60:.1f} min

**Training curve**
| Epoch | Train Loss | Val Loss | Val micro-F1 | Val macro-F1 |
|---|---|---|---|---|
{chr(10).join(curve_rows)}

**Test results**
| threshold | micro-F1 | macro-F1 |
|---|---|---|
| tuned | {metrics_tuned['micro_f1']:.4f} | {metrics_tuned['macro_f1']:.4f} |
| flat 0.5 | {metrics_flat['micro_f1']:.4f} | {metrics_flat['macro_f1']:.4f} |

**Bottom 10 classes by F1**
{bottom10}

**Top 10 classes by F1**
{top10}

---
"""
    summary_path = RESULTS / "SUMMARY.md"
    with summary_path.open("a" if summary_path.exists() else "w", encoding="utf-8") as f:
        f.write(summary)
    print(f"SUMMARY.md updated → {summary_path}")

    # free memory before next run
    del clf, trainer, train_dataset, val_dataset, test_dataset
    del train_chunks, val_chunks, test_chunks
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics_tuned

print("run_one() defined.")

## 5. Run

In [ ]:
all_results = []

if SWEEP_MODE:
    sweep_files = sorted(SWEEPS_DIR.glob("*.json"))
    if RUN_ONLY:
        sweep_files = [f for f in sweep_files if f.name in RUN_ONLY]
    print(f"Sweep mode — running: {[f.name for f in sweep_files]}")

    for sweep_file in sweep_files:
        try:
            cfg     = load_bert_config(sweep_file)
            metrics = run_one(cfg, run_name=sweep_file.stem)
            all_results.append({
                "run"      : sweep_file.stem,
                "model"    : cfg.model_name,
                "pooling"  : cfg.pooling,
                "loss"     : cfg.loss_type,
                "chunks"   : cfg.max_chunks,
                "micro_f1" : metrics["micro_f1"],
                "macro_f1" : metrics["macro_f1"],
            })
        except Exception as e:
            print(f"✗ {sweep_file.stem} FAILED: {e}")
            traceback.print_exc()
            all_results.append({"run": sweep_file.stem, "error": str(e)})
else:
    # single run from bert_config.json
    metrics = run_one(BERT_CFG, run_name="bert")
    all_results.append({
        "run"      : "bert",
        "model"    : BERT_CFG.model_name,
        "pooling"  : BERT_CFG.pooling,
        "loss"     : BERT_CFG.loss_type,
        "chunks"   : BERT_CFG.max_chunks,
        "micro_f1" : metrics["micro_f1"],
        "macro_f1" : metrics["macro_f1"],
    })

# final comparison table
results_df = pd.DataFrame(all_results)
print(f"\n{'='*60}")
print("ALL RESULTS")
print(results_df[[c for c in ["run","micro_f1","macro_f1","error"] if c in results_df.columns]].to_string(index=False))
results_df.to_csv(RESULTS / "sweep_results.csv", index=False)
print(f"Saved → results/sweep_results.csv")